# Sentiment Style Transfer — Negative → Positive
### Delete, Retrieve, Generate (Li, Jia, He & Liang — NAACL 2018)

Run **`Twitter_Sentiment_NLP_updated.ipynb` first** and bring `project_bundle.zip` here.

Input: tweets your classifier labelled **Negative**.
Output: the same tweets, same content, **Positive** tone.

```
DELETE    strip the negative attribute markers        -> content template
RETRIEVE  nearest positive tweet by content           -> borrow its positive markers
GENERATE  4 variants:
            (a) DeleteOnly     template as-is                       [paper baseline]
            (b) TemplateBased  splice borrowed markers in           [paper baseline]
            (c) Gemini         fluent rewrite from template+markers [the LLM generator]
            (d) Gemini+Rerank  k candidates, pick argmax p_pos x content_sim
EVALUATE  style accuracy (your Part-1 classifier) + self-BLEU (content preservation)
```

(c) and (d) are what "Transforming DRG" (Sudhakar et al., EMNLP 2019) does with GPT-2 —
the same skeleton, a stronger generator in the GENERATE slot. (d) is Prompt-and-Rerank
(Suzgun et al., EMNLP 2022) with your own LogReg as the style scorer.

Sections (a) and (b) need **no API key** and will run offline.

## 0. Setup

In [ ]:
!pip install -q google-genai

In [ ]:
import os
import re
import json
import time
import pickle
import zipfile

import numpy as np
import pandas as pd

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# --- config ------------------------------------------------------------------
PROJECT_DIR    = 'project'
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')   # <- paste here if not using an env var
GEMINI_MODEL   = 'gemini-2.5-flash'

N_CANDIDATES   = 4     # rewrites Gemini produces per tweet (for the reranker)
BATCH_SIZE     = 20    # tweets per API call -- keeps you well inside the free-tier RPM
N_EVAL         = 100   # held-out negatives to score in the results table

TOKEN_PATTERN  = r"(?u)\b\w+\b"   # must match notebook 1 exactly

### Upload `project_bundle.zip`

On Colab, run the next cell and pick the zip. Running locally with a `project/` folder already
next to this notebook? Just skip it.

In [ ]:
if not os.path.isdir(PROJECT_DIR):
    try:
        from google.colab import files
        up = files.upload()                      # choose project_bundle.zip
        with zipfile.ZipFile(list(up.keys())[0]) as z:
            z.extractall(PROJECT_DIR)
    except ImportError:
        raise SystemExit('Put the project/ folder next to this notebook first.')

print(sorted(os.listdir(PROJECT_DIR)))

## 1. Load the artifacts

In [ ]:
# The Part-1 classifier. Reused here purely as an EVALUATOR / reranker -- never retrained.
model      = pickle.load(open(f'{PROJECT_DIR}/trained_model.sav', 'rb'))
vectorizer = pickle.load(open(f'{PROJECT_DIR}/vectorizer.sav', 'rb'))

# The DRG artifacts.
markers = json.load(open(f'{PROJECT_DIR}/drg_markers.json'))
neg_markers = set(markers['neg_markers'])
pos_markers = set(markers['pos_markers'])

drg = pickle.load(open(f'{PROJECT_DIR}/drg_retrieval.sav', 'rb'))
retrieval_vectorizer = drg['retrieval_vectorizer']
pos_template_matrix  = drg['pos_template_matrix']
pos_templates        = drg['pos_templates']
pos_template_markers = drg['pos_template_markers']
pos_corpus           = drg['pos_corpus']

print(f'negative markers : {len(neg_markers)}')
print(f'positive markers : {len(pos_markers)}')
print(f'retrieval index  : {pos_template_matrix.shape}')

## 2. The DRG functions (identical to notebook 1)

In [ ]:
port_stem  = PorterStemmer()
stop_words = set(stopwords.words('english'))


def stemming(content):
    """Exact replica of notebook 1's stemming(). Only used to feed the CLASSIFIER."""
    stemmed_content = re.sub('[^a-zA-Z]', ' ', content)
    stemmed_content = stemmed_content.lower().split()
    stemmed_content = [port_stem.stem(w) for w in stemmed_content if w not in stop_words]
    return ' '.join(stemmed_content)


def light_clean(content):
    """Exact replica of notebook 1's light_clean(). Used for DELETE / RETRIEVE."""
    content = content.lower()
    content = re.sub(r'http\S+|www\.\S+', ' ', content)
    content = re.sub(r'@\w+', ' ', content)
    content = re.sub(r"'", '', content)
    content = re.sub(r'[^a-z\s]', ' ', content)
    content = re.sub(r'\s+', ' ', content).strip()
    return content


def delete_markers(sentence, marker_set):
    """DELETE. Bigrams greedily first, then leftover unigrams."""
    words = sentence.split()
    n = len(words)
    drop = [False] * n
    found = []

    i = 0
    while i < n - 1:
        bg = words[i] + ' ' + words[i + 1]
        if bg in marker_set:
            drop[i] = drop[i + 1] = True
            found.append(bg)
            i += 2
        else:
            i += 1

    for i in range(n):
        if not drop[i] and words[i] in marker_set:
            drop[i] = True
            found.append(words[i])

    template = ' '.join(w for i, w in enumerate(words) if not drop[i])
    return template, found


def retrieve(template, top_n=1):
    """RETRIEVE. Nearest positive tweet by content -> its positive markers."""
    q = retrieval_vectorizer.transform([template])
    sims = (pos_template_matrix @ q.T).toarray().ravel()
    idx = np.argsort(sims)[::-1][:top_n]
    return [{'markers':   pos_template_markers[i],
             'neighbour': pos_corpus[i],
             'sim':       float(sims[i])} for i in idx]


def prepare(tweet, tweet_id=0):
    """DELETE + RETRIEVE. Everything the GENERATE step needs, for one tweet."""
    clean = light_clean(tweet)
    template, deleted = delete_markers(clean, neg_markers)
    hits = retrieve(template)
    borrowed = hits[0]['markers'][:3] if hits else []
    return {
        'id':            tweet_id,
        'input':         tweet,
        'template':      template,
        'deleted':       deleted,
        'borrowed':      borrowed,
        'neighbour':     hits[0]['neighbour'] if hits else '',
        'sim':           hits[0]['sim'] if hits else 0.0,
        'delete_only':   template,
        'template_based': (' '.join(borrowed[:2]) + ' ' + template).strip(),
    }

## 3. Baselines — no API key needed

In [ ]:
DEMO_NEGATIVES = [
    "the battery on this phone is terrible and the support team is useless",
    "worst customer service ever, waited 2 hours and nobody even helped me",
    "this update broke everything, so frustrated right now",
    "really disappointed with the new season, such a waste of time",
    "my flight got delayed again, i hate this airline so much",
]

for i, t in enumerate(DEMO_NEGATIVES):
    d = prepare(t, i)
    print('IN            :', d['input'])
    print('  deleted     :', d['deleted'])
    print('  template    :', d['template'])
    print(f"  borrowed    : {d['borrowed']}   (sim={d['sim']:.2f})")
    print('  neighbour   :', d['neighbour'][:70])
    print('  DeleteOnly  :', d['delete_only'])
    print('  TemplateBased:', d['template_based'])
    print()

Those outputs are meant to look broken. `TemplateBased` is the paper's model-free variant and it
is the whole argument for putting a real generator in the GENERATE slot — keep the numbers, they
are your baseline row.

## 4. GENERATE with Gemini

Batched: `BATCH_SIZE` tweets per request, `N_CANDIDATES` rewrites each. 100 tweets = 5 calls,
comfortably inside the free tier.

In [ ]:
from google import genai
from google.genai import types
from pydantic import BaseModel


class Rewrite(BaseModel):
    id: int
    rewrites: list[str]


REWRITE_PROMPT = """You are doing sentiment style transfer on tweets: NEGATIVE -> POSITIVE.

For each item you get:
- "original"  : the negative tweet
- "template"  : the same tweet with its negative sentiment words deleted. This is the CONTENT
                that must survive the rewrite.
- "borrowed"  : positive sentiment words lifted from a real positive tweet about similar content

Rewrite each tweet so that:
- the sentiment is clearly POSITIVE
- every topic, entity and fact in "template" is preserved -- do not invent new facts, do not drop any
- the "borrowed" words are worked in where they fit naturally (ignore any that do not fit)
- it still reads like a real tweet: casual, under 280 characters

Give {k} DIFFERENT rewrites per item.
Return ONLY a JSON array of objects with keys: id, rewrites.

Items:
{items}
"""


def gemini_rewrite(records, k=N_CANDIDATES, batch_size=BATCH_SIZE, verbose=True):
    """records: list of dicts from prepare(). Returns {id: [candidate, ...]}."""
    client = genai.Client(api_key=GEMINI_API_KEY)
    out = {}

    for start in range(0, len(records), batch_size):
        chunk = records[start:start + batch_size]
        items = json.dumps(
            [{'id': r['id'], 'original': r['input'], 'template': r['template'],
              'borrowed': r['borrowed']} for r in chunk],
            ensure_ascii=False, indent=1,
        )
        prompt = REWRITE_PROMPT.format(k=k, items=items)

        for attempt in range(3):
            try:
                resp = client.models.generate_content(
                    model=GEMINI_MODEL,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        response_mime_type='application/json',
                        response_schema=list[Rewrite],
                        temperature=1.0,
                    ),
                )
                for d in json.loads(resp.text):
                    cands = [str(c).strip() for c in d.get('rewrites', []) if str(c).strip()]
                    if cands:
                        out[int(d['id'])] = cands
                break
            except Exception as e:
                if verbose:
                    print(f'  batch {start}: attempt {attempt + 1} failed ({e}); retrying')
                time.sleep(5 * (attempt + 1))

        if verbose:
            print(f'  {min(start + batch_size, len(records))}/{len(records)} done')

    return out

### The reranker

`Prompt-and-Rerank`, with your Part-1 classifier as the style scorer. Two signals:

- **style**   `P(positive)` from the LogReg — how far the rewrite actually moved
- **content** cosine similarity in the *retrieval* vocabulary, which was fitted on
  sentiment-stripped templates. So it scores the content words and mostly ignores the sentiment
  words — which is exactly what "content preservation" is supposed to mean.

`score = P(positive) x content_sim`.

In [ ]:
def positivity(texts):
    """P(positive) from the Part-1 classifier."""
    X = vectorizer.transform([stemming(t) for t in texts])
    return model.predict_proba(X)[:, 1]


def content_sim(original, candidate):
    va = retrieval_vectorizer.transform([light_clean(original)])
    vb = retrieval_vectorizer.transform([light_clean(candidate)])
    return float((va @ vb.T).toarray().ravel()[0])


def rerank(original, candidates):
    p    = positivity(candidates)
    sims = np.array([content_sim(original, c) for c in candidates])
    scores = p * sims
    best = int(np.argmax(scores))
    return candidates[best], {
        'p_pos': float(p[best]),
        'sim':   float(sims[best]),
        'score': float(scores[best]),
        'all':   [(c, round(float(pp), 3), round(float(ss), 3))
                  for c, pp, ss in zip(candidates, p, sims)],
    }


def style_transfer(tweets):
    """Full pipeline. Returns one record per tweet with all four outputs."""
    records = [prepare(t, i) for i, t in enumerate(tweets)]

    if GEMINI_API_KEY:
        cands = gemini_rewrite(records)
        for r in records:
            c = cands.get(r['id'], [])
            if c:
                r['gemini'] = c[0]                        # first candidate, no reranking
                r['gemini_rerank'], r['rerank_info'] = rerank(r['input'], c)
                r['candidates'] = c
            else:                                          # API hiccup -> fall back, never crash
                r['gemini'] = r['template_based']
                r['gemini_rerank'] = r['template_based']
                r['rerank_info'] = {}
                r['candidates'] = []
    else:
        print('No GEMINI_API_KEY set -- baselines only.')
        for r in records:
            r['gemini'] = r['template_based']
            r['gemini_rerank'] = r['template_based']
            r['rerank_info'] = {}
            r['candidates'] = []

    return records

## 5. Run it on the demo tweets

In [ ]:
demo = style_transfer(DEMO_NEGATIVES)

for d in demo:
    print('NEGATIVE  :', d['input'])
    print('  deleted :', d['deleted'], '| borrowed:', d['borrowed'])
    print('  Template:', d['template_based'])
    print('  Gemini  :', d['gemini'])
    print('  +Rerank :', d['gemini_rerank'])
    if d['rerank_info']:
        print(f"            (p_pos={d['rerank_info']['p_pos']:.2f}  sim={d['rerank_info']['sim']:.2f})")
    print()

In [ ]:
# What the reranker actually threw away -- worth a screenshot for the report.
d = demo[0]
if d['rerank_info']:
    print('ORIGINAL:', d['input'], '\n')
    for c, p, s in sorted(d['rerank_info']['all'], key=lambda x: -x[1] * x[2]):
        print(f'  p_pos={p:.2f}  sim={s:.2f}  score={p * s:.3f}  |  {c}')

## 6. Evaluation

On the **held-out** negatives saved by notebook 1 (not in the marker sample).

- **style accuracy** — fraction the Part-1 classifier now calls Positive. Higher = better transfer.
- **self-BLEU** — BLEU of the output against the *input*. Higher = more content preserved.

They trade off. A model that answers `"i love it"` to everything scores style 1.00 and BLEU ~0.
A model that copies the input scores BLEU 1.00 and style ~0. The point is to be good at both.

In [ ]:
eval_negatives = json.load(open(f'{PROJECT_DIR}/eval_negatives.json'))[:N_EVAL]
print(len(eval_negatives), 'held-out negative tweets')
for t in eval_negatives[:3]:
    print(' -', t)

In [ ]:
eval_records = style_transfer(eval_negatives)
print('done')

In [ ]:
smooth = SmoothingFunction().method1


def evaluate(name, records, field):
    pairs = [(r['input'], r[field]) for r in records if r[field].strip()]
    outs  = [o for _, o in pairs]

    preds = model.predict(vectorizer.transform([stemming(o) for o in outs]))
    style_acc = float(np.mean(preds == 1))

    bleu = float(np.mean([
        sentence_bleu([light_clean(i).split()], light_clean(o).split(), smoothing_function=smooth)
        for i, o in pairs
    ]))

    return {'method': name, 'style_acc': round(style_acc, 3), 'self_bleu': round(bleu, 3),
            'n': len(pairs)}


rows = [
    evaluate('Copy input (floor)', [{**r, 'copy': r['input']} for r in eval_records], 'copy'),
    evaluate('DeleteOnly',         eval_records, 'delete_only'),
    evaluate('TemplateBased',      eval_records, 'template_based'),
]
if GEMINI_API_KEY:
    rows += [
        evaluate('DRG + Gemini',          eval_records, 'gemini'),
        evaluate('DRG + Gemini + Rerank', eval_records, 'gemini_rerank'),
    ]

results = pd.DataFrame(rows)
results

Read the table like this: `Copy input` pins the top of the BLEU column and the bottom of the style
column. Every row below it is buying style accuracy with content. DRG + Gemini + Rerank should sit
highest on style while holding BLEU well above `TemplateBased` — that gap is the result you report.

## 7. Save

In [ ]:
out = pd.DataFrame([{
    'input':          r['input'],
    'deleted':        ' | '.join(r['deleted']),
    'template':       r['template'],
    'borrowed':       ' | '.join(r['borrowed']),
    'neighbour':      r['neighbour'],
    'delete_only':    r['delete_only'],
    'template_based': r['template_based'],
    'gemini':         r['gemini'],
    'gemini_rerank':  r['gemini_rerank'],
} for r in eval_records])

out.to_csv('style_transfer_outputs.csv', index=False)
results.to_csv('style_transfer_results.csv', index=False)

print('style_transfer_outputs.csv :', out.shape)
print('style_transfer_results.csv :', results.shape)
out.head(10)

## Notes for the report

**What this is.** Delete-Retrieve-Generate (Li et al., NAACL 2018) with the DELETE step driven by
the LogisticRegression coefficients + Li et al.'s salience ratio, and the GENERATE step served by
an LLM instead of the paper's LSTM — i.e. the "Transforming DRG" idea (Sudhakar et al., EMNLP 2019),
plus classifier-guided reranking (Suzgun et al., EMNLP 2022).

**Honest limitations, put them in the report before someone asks:**
- Style accuracy is scored by the same classifier family that produced the markers. It is
  self-referential and will read a little high. A held-out neutral judge (a BERT sentiment model,
  or Gemini itself as a judge) would be the fix.
- self-BLEU against the input rewards copying. Real DRG papers also report BLEU against
  *human references*; Sentiment140 has none, which is why the Yelp benchmark exists.
- No fluency metric. Add GPT-2 perplexity if you want the third column of the standard table.

**Cite:**
- Li, Jia, He, Liang. *Delete, Retrieve, Generate: A Simple Approach to Sentiment and Style Transfer.* NAACL 2018.
- Sudhakar, Upadhyay, Maheswaran. *"Transforming" Delete, Retrieve, Generate Approach for Controlled Text Style Transfer.* EMNLP 2019.
- Suzgun, Melas-Kyriazi, Jurafsky. *Prompt-and-Rerank.* EMNLP 2022.